# Exploratory Data Analysis (EDA)

## NYC Green Taxi Trip Records (April & Mey 2026)

This notebook explores the NYC Green Taxi trip dataset to understand its structure, assess data quality, identify potential anomalies, and define data cleaning rules for the ETL pipeline.

In [ ]:
import pandas as pd
import pyarrow as pa

In [2]:
df_april = pd.read_parquet("/Users/agungnugraha/Code/cloud-data-pipeline/data/raw/green_tripdata_2026-04.parquet")
df_may = pd.read_parquet("/Users/agungnugraha/Code/cloud-data-pipeline/data/raw/green_tripdata_2026-05.parquet")

df = pd.concat([df_april, df_may], ignore_index=True)

In [3]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2026-04-01 00:29:09,2026-04-01 00:54:48,N,1.0,66,48,1.0,6.56,31.0,...,0.5,11.10,0.0,NaN,1.0,48.10,1.0,1.0,2.75,0.75
1,2,2026-04-01 00:19:13,2026-04-01 00:19:20,N,5.0,129,129,1.0,0.00,26.0,...,0.0,0.00,0.0,NaN,1.0,27.00,1.0,2.0,0.00,0.00
2,2,2026-04-01 00:40:24,2026-04-01 00:47:31,N,1.0,74,141,1.0,2.56,12.1,...,0.5,3.47,0.0,NaN,1.0,20.82,1.0,1.0,2.75,0.00
3,2,2026-04-01 00:27:31,2026-04-01 00:35:33,N,1.0,244,116,1.0,1.48,10.0,...,0.5,0.00,0.0,NaN,1.0,12.50,2.0,1.0,0.00,0.00
4,2,2026-04-01 00:51:45,2026-04-01 00:58:33,N,1.0,243,235,1.0,0.93,8.6,...,0.5,0.01,0.0,NaN,1.0,11.11,1.0,1.0,0.00,0.00


In [4]:
print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

Rows    : 89,159
Columns : 21


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 89159 entries, 0 to 89158
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               89159 non-null  int32         
 1   lpep_pickup_datetime   89159 non-null  datetime64[us]
 2   lpep_dropoff_datetime  89159 non-null  datetime64[us]
 3   store_and_fwd_flag     77097 non-null  str           
 4   RatecodeID             77097 non-null  float64       
 5   PULocationID           89159 non-null  int32         
 6   DOLocationID           89159 non-null  int32         
 7   passenger_count        77097 non-null  float64       
 8   trip_distance          89159 non-null  float64       
 9   fare_amount            89159 non-null  float64       
 10  extra                  89159 non-null  float64       
 11  mta_tax                89159 non-null  float64       
 12  tip_amount             89159 non-null  float64       
 13  tolls_amount

In [10]:
schema = pa.Table.from_pandas(df)
print(schema.schema)

VendorID: int32
lpep_pickup_datetime: timestamp[us]
lpep_dropoff_datetime: timestamp[us]
store_and_fwd_flag: large_string
RatecodeID: double
PULocationID: int32
DOLocationID: int32
passenger_count: double
trip_distance: double
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
ehail_fee: double
improvement_surcharge: double
total_amount: double
payment_type: double
trip_type: double
congestion_surcharge: double
cbd_congestion_fee: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 2948


In [11]:
missing_summary = pd.DataFrame({
    "Missing_value": df.isna().sum(),
    "Percentage": (df.isna().sum() / len(df)) * 100,
}).sort_values(by="Missing_value", ascending=False)

missing_summary

,Missing_value,Percentage
ehail_fee,89159,100.00000
congestion_surcharge,12062,13.52864
store_and_fwd_flag,12062,13.52864
RatecodeID,12062,13.52864
trip_type,12062,13.52864
payment_type,12062,13.52864
passenger_count,12062,13.52864
VendorID,0,0.00000
tip_amount,0,0.00000
total_amount,0,0.00000


In [12]:
print("Total data duplicate:", df.duplicated().sum())

Total data duplicate: 0


In [13]:
pd.DataFrame({
    "unique": df.nunique()
}).sort_values("unique")

,unique
ehail_fee,0
trip_type,2
store_and_fwd_flag,2
VendorID,3
cbd_congestion_fee,3
congestion_surcharge,4
payment_type,4
improvement_surcharge,5
RatecodeID,5
mta_tax,6


In [14]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
VendorID,89159.0,2.324903,1.0,2.0,2.0,2.0,6.0,1.258009
lpep_pickup_datetime,89159,2026-05-01 04:51:59.235052,2008-12-31 23:05:50,2026-04-16 15:31:44.500000,2026-05-01 09:48:55,2026-05-15 18:11:57,2026-05-31 23:59:13,NaN
lpep_dropoff_datetime,89159,2026-05-01 05:13:46.461748,2008-12-31 23:31:29,2026-04-16 15:56:21,2026-05-01 10:06:52,2026-05-15 18:35:23,2026-06-01 21:06:14,NaN
RatecodeID,77097.0,1.258026,1.0,1.0,1.0,1.0,5.0,0.97394
PULocationID,89159.0,98.303312,1.0,74.0,75.0,116.0,265.0,57.289433
DOLocationID,89159.0,144.229837,1.0,75.0,141.0,231.0,265.0,77.237427
passenger_count,77097.0,1.307249,0.0,1.0,1.0,1.0,9.0,0.963912
trip_distance,89159.0,12.894934,0.0,1.27,2.07,3.69,111005.95,801.820236
fare_amount,89159.0,17.220803,-250.08,8.6,13.5,19.8,670.1,18.23839
extra,89159.0,0.821813,-2.5,0.0,0.0,1.0,7.5,1.370549


### Distribution Analysis

In [22]:
low_card_cols = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type",
    "trip_type",
    "passenger_count",
]

for col in low_card_cols:
    summary = pd.DataFrame({
        "count": df[col].value_counts(dropna=False),
        "percentage (%)": (df[col].value_counts(dropna=False, normalize=True) * 100).round(2)
    }).sort_index()
    
    print(col)
    print(summary)

VendorID
          count  percentage (%)
VendorID                       
1          6928            7.77
2         73257           82.16
6          8974           10.07
RatecodeID
            count  percentage (%)
RatecodeID                       
1.0         71874           80.61
2.0           262            0.29
3.0            58            0.07
4.0            97            0.11
5.0          4806            5.39
NaN         12062           13.53
store_and_fwd_flag
                    count  percentage (%)
store_and_fwd_flag                       
N                   77001           86.36
Y                      96            0.11
NaN                 12062           13.53
payment_type
              count  percentage (%)
payment_type                       
1.0           59018           66.19
2.0           17441           19.56
3.0             441            0.49
4.0             197            0.22
NaN           12062           13.53
trip_type
           count  percentage (%)
trip_type  

In [24]:
high_card_cols = ["PULocationID", "DOLocationID"]

for col in high_card_cols:
    print(col)
    top5 = (df[col].value_counts(dropna=False, normalize=True).head(10) * 100).round(2)
    
    summary = pd.DataFrame({
        "count": df[col].value_counts(dropna=False).head(5),
        "percentage (%)": top5
    })
    print(summary)
    print(f"Total Unique Zones Active: {df[col].nunique()}")

PULocationID
                count  percentage (%)
PULocationID                         
41                NaN            3.11
43             3468.0            3.89
65                NaN            3.49
74            23586.0           26.45
75            11257.0           12.63
82                NaN            3.56
95             4073.0            4.57
97                NaN            2.60
130               NaN            2.55
166            3805.0            4.27
Total Unique Zones Active: 243
DOLocationID
               count  percentage (%)
DOLocationID                        
41               NaN            3.02
42               NaN            2.73
74            4054.0            4.55
75            5177.0            5.81
138              NaN            2.59
166           2946.0            3.30
236           4942.0            5.54
238           3416.0            3.83
239              NaN            2.32
263              NaN            2.67
Total Unique Zones Active: 251


In [20]:
fare_cols = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
]

fare_summary = df[fare_cols].agg(["count", "mean", "median", "min", "max"]).T
print(fare_summary)

                         count       mean  median     min     max
fare_amount            89159.0  17.220803   13.50 -250.08  670.10
extra                  89159.0   0.821813    0.00   -2.50    7.50
mta_tax                89159.0   0.539805    0.50   -0.50    5.00
tip_amount             89159.0   2.698872    2.02   -1.18  470.00
tolls_amount           89159.0   0.281365    0.00    0.00   48.50
improvement_surcharge  89159.0   0.917535    1.00   -1.00    1.00
congestion_surcharge   77097.0   0.894785    0.00   -2.75    2.75


### Datetime Validation

In [26]:
pickup_start = pd.Timestamp("2026-04-01")
pickup_end = pd.Timestamp("2026-05-31 23:59:59")

pickup_out_of_range = (df["lpep_pickup_datetime"] < pickup_start) | (df["lpep_pickup_datetime"] > pickup_end)

total_out_of_range = pickup_out_of_range.sum()
print("Pickup outside reporting period (Apr-May 2026):", total_out_of_range)

if total_out_of_range > 0:
    print("\nOutside Periode")
    out_of_range_summary = (
        df.loc[pickup_out_of_range, "lpep_pickup_datetime"]
        .dt.strftime("%Y-%m")
        .value_counts()
        .sort_index()
    )
    print(out_of_range_summary)

Pickup outside reporting period (Apr-May 2026): 3

Outside Periode
lpep_pickup_datetime
2008-12    2
2026-03    1
Name: count, dtype: int64


In [27]:
df.loc[pickup_out_of_range, ["lpep_pickup_datetime", "lpep_dropoff_datetime"]]

,lpep_pickup_datetime,lpep_dropoff_datetime
18,2026-03-31 23:28:50,2026-04-01 23:17:31
54149,2008-12-31 23:12:44,2008-12-31 23:31:29
59191,2008-12-31 23:05:50,2009-01-01 20:42:01


In [28]:
# make a new column 
df["trip_duration_minutes"] = (df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]).dt.total_seconds() / 60

In [29]:
zero_duration = df["trip_duration_minutes"] == 0
invalid_time_order = df["trip_duration_minutes"] < 0

print("Zero-duration trips:", zero_duration.sum())
print("Dropoff before pickup:", invalid_time_order.sum())

Zero-duration trips: 65
Dropoff before pickup: 0


In [33]:
zero_distance = (df["trip_distance"] == 0)
negative_distance = (df["trip_distance"] < 0)

zero_distance_df = (df.loc[zero_distance].copy())


print(f"Negative-distance trips: {negative_distance.sum():,}")
print(f"Zero-distance trips: {len(zero_distance_df):,}")

Negative-distance trips: 0
Zero-distance trips: 3,164


In [34]:
zero_duration = (zero_distance_df["trip_duration_minutes"] == 0)
positive_duration = (zero_distance_df["trip_duration_minutes"] > 0)

print(f"Zero-distance trips with zero duration: {zero_duration.sum():,}")
print(f"Zero-distance trips with positive duration: {positive_duration.sum():,}")

Zero-distance trips with zero duration: 58
Zero-distance trips with positive duration: 3,106


In [35]:
zero_total = (zero_distance_df["total_amount"] == 0)
positive_total = (zero_distance_df["total_amount"] > 0)
negative_total = (zero_distance_df["total_amount"] < 0)

print(f"Zero-distance trips with zero total amount: {zero_total.sum():,}")
print(f"Zero-distance trips with positive total amount: {positive_total.sum():,}")
print(f"Zero-distance trips with negative total amount: {negative_total.sum():,}")

Zero-distance trips with zero total amount: 23
Zero-distance trips with positive total amount: 3,005
Zero-distance trips with negative total amount: 136


In [36]:
suspicious_zero_distance = (zero_distance_df["trip_duration_minutes"] == 0) & (zero_distance_df["total_amount"] > 0)
print(f"Zero-distance trips with zero duration and positive total amount: {suspicious_zero_distance.sum():,}")

Zero-distance trips with zero duration and positive total amount: 54


In [37]:
zero_distance_df.loc[
    suspicious_zero_distance,
    [
        "trip_distance",
        "trip_duration_minutes",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "payment_type",
    ],
]

,trip_distance,trip_duration_minutes,fare_amount,tip_amount,total_amount,payment_type
42,0.0,0.0,17.00,0.00,17.00,2.0
121,0.0,0.0,17.00,0.00,17.00,2.0
1040,0.0,0.0,5.00,5.08,10.08,1.0
1297,0.0,0.0,3.00,35.00,40.50,1.0
2753,0.0,0.0,10.00,2.00,12.00,1.0
3145,0.0,0.0,3.00,0.00,7.00,2.0
3229,0.0,0.0,3.70,0.00,7.70,2.0
5279,0.0,0.0,3.00,0.00,4.50,2.0
5791,0.0,0.0,12.72,0.00,12.72,1.0
7186,0.0,0.0,36.60,0.00,40.85,2.0


### Business Validation Rules

In [38]:
print(f"Negative passenger count: {(df['passenger_count'] < 0).sum():,}")
print(f"Zero passenger count: {(df['passenger_count'] == 0).sum():,}")

Negative passenger count: 0
Zero passenger count: 1,143


In [39]:
money_columns = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "cbd_congestion_fee",
]

for column in money_columns:
    print(f"{column}: {(df[column] < 0).sum():,} negative values")

fare_amount: 273 negative values
extra: 94 negative values
mta_tax: 228 negative values
tip_amount: 15 negative values
tolls_amount: 0 negative values
improvement_surcharge: 278 negative values
total_amount: 278 negative values
congestion_surcharge: 4 negative values
cbd_congestion_fee: 2 negative values


In [40]:
calculated_total = (
    df["fare_amount"]
    + df["extra"]
    + df["mta_tax"]
    + df["tip_amount"]
    + df["tolls_amount"]
    + df["improvement_surcharge"]
    + df["congestion_surcharge"].fillna(0)
    + df["cbd_congestion_fee"].fillna(0)
)

total_mismatch = ((calculated_total - df["total_amount"]).round(2) != 0)
print(f"Total amount mismatch: {total_mismatch.sum():,}")

Total amount mismatch: 16,687


In [41]:
cash_with_tip = ((df["payment_type"] == 2) & (df["tip_amount"] > 0))
print(f"Cash payment with tip: {cash_with_tip.sum():,}")

Cash payment with tip: 0


In [42]:
print("Pickup location")
print(f"Minimum: {df['PULocationID'].min()}")
print(f"Maximum: {df['PULocationID'].max()}\n")

print("Dropoff location")
print(f"Minimum: {df['DOLocationID'].min()}")
print(f"Maximum: {df['DOLocationID'].max()}")

Pickup location
Minimum: 1
Maximum: 265

Dropoff location
Minimum: 1
Maximum: 265


### Join Simulation Analysis

In [43]:
df_zone = pd.read_csv("/Users/agungnugraha/Code/nyc-taxi-pipeline-gcp-native/data/raw/taxi_zone_lookup.csv")

In [44]:
df = df.merge(df_zone, left_on="PULocationID", right_on="LocationID", how="left")

In [45]:
missing_pickup_zone = (df["Borough"].isnull().sum())
print(f"Trips with missing pickup zone: {missing_pickup_zone:,}")

Trips with missing pickup zone: 72


In [46]:
df.loc[df["Borough"].isna(),"PULocationID"].value_counts()

PULocationID
265    72
Name: count, dtype: int64

In [47]:
df_zone.loc[df_zone["LocationID"] == 265]

,LocationID,Borough,Zone,service_zone
264,265,NaN,Outside of NYC,NaN


In [48]:
invalid_locations_list = [264, 265]

is_valid_location = (
    ~df['PULocationID'].isin(invalid_locations_list) & 
    ~df['DOLocationID'].isin(invalid_locations_list) &
    df['PULocationID'].notna() &
    df['DOLocationID'].notna()
)

total_rows = len(df)
valid_rows = is_valid_location.sum()
failed_rows = (~is_valid_location).sum()
failed_pct = (failed_rows / total_rows) * 100

print(f"Total Rows              : {total_rows}")
print(f"Valid Location Rows     : {valid_rows}")
print(f"Failed Location Rows    : {failed_rows}")  
print(f"Failed Percentage       : {failed_pct:.2f}%")

Total Rows              : 89159
Valid Location Rows     : 87558
Failed Location Rows    : 1601
Failed Percentage       : 1.80%


In [54]:
invalid_pu = df["PULocationID"].isin(invalid_locations_list)
invalid_do = df["DOLocationID"].isin(invalid_locations_list)

invalid_location_mask = invalid_pu | invalid_do
total_invalid_location = invalid_location_mask.sum()

print(f"Trips with invalid location (264/265 in PU or DO): {total_invalid_location:,}")

print(f"Invalid in PULocationID only : {(invalid_pu & ~invalid_do).sum():,}")
print(f"Invalid in DOLocationID only : {(~invalid_pu & invalid_do).sum():,}")
print(f"Invalid in BOTH (PU & DO)    : {(invalid_pu & invalid_do).sum():,}")

Trips with invalid location (264/265 in PU or DO): 1,601
Invalid in PULocationID only : 57
Invalid in DOLocationID only : 1,297
Invalid in BOTH (PU & DO)    : 247
